In [1]:
# ==========================================
# Notebook 1: Read & Join Tables
# ==========================================

import os
import pandas as pd
from sqlalchemy import create_engine

# 1. Création du dossier pour conserver les artefacts
os.makedirs("artifacts", exist_ok=True)

In [5]:
# 1. Configuration de la connexion avec SQLAlchemy
SERVER = 'LOCALHOST'
DATABASE = 'MLops_Training'

connection_url = (
    f"mssql+pyodbc://@{SERVER}/{DATABASE}?"
    f"driver=ODBC+Driver+17+for+SQL+Server&"
    f"trusted_connection=yes&"
    f"TrustServerCertificate=yes&"
    f"login_timeout=5"
)

engine = create_engine(connection_url)

# 2. Test rapide de lecture
print("Lecture de la table orders...")

# 3. Lecture des tables (sans aucun avertissement)
orders = pd.read_sql_query("SELECT * FROM [MLops_Training].[Olist].olist_orders_dataset", engine)
#print(orders)
customers = pd.read_sql_query("SELECT * FROM [MLops_Training].[Olist].olist_customers_dataset", engine)
order_items = pd.read_sql_query("SELECT * FROM [MLops_Training].[Olist].olist_order_items_dataset", engine)
payments = pd.read_sql_query("SELECT * FROM [MLops_Training].[Olist].olist_order_payments_dataset", engine)
products = pd.read_sql_query("SELECT * FROM [MLops_Training].[Olist].olist_products_dataset", engine)
sellers = pd.read_sql_query("SELECT * FROM [MLops_Training].[Olist].olist_sellers_dataset", engine)
print("Toutes les tables sont chargées !")

Lecture de la table orders...
Toutes les tables sont chargées !


In [6]:
# 1. Inspection rapide (Row counts, Clés & Doublons)
tables = {
    "orders": (orders, "order_id"),
    "customers": (customers, "customer_unique_id"),
    "order_items": (order_items, "order_item_id"),
    "payments": (payments, "order_id"),
    "products": (products, "product_id"),
    "sellers": (sellers, "seller_id")
}

print("\n=== INSPECTION DES TABLES ===")
for name, (df, pkey) in tables.items():
    unique_keys = df[pkey].nunique() if pkey in df.columns else 'N/A'
    print(f"Table '{name}': {len(df)} rows | Duplicates: {df.duplicated().sum()} | Unique keys ({pkey}): {unique_keys}")


=== INSPECTION DES TABLES ===
Table 'orders': 99441 rows | Duplicates: 0 | Unique keys (order_id): 99441
Table 'customers': 99441 rows | Duplicates: 0 | Unique keys (customer_unique_id): 96096
Table 'order_items': 112650 rows | Duplicates: 0 | Unique keys (order_item_id): 21
Table 'payments': 103886 rows | Duplicates: 0 | Unique keys (order_id): 99440
Table 'products': 32951 rows | Duplicates: 0 | Unique keys (product_id): 32951
Table 'sellers': 3095 rows | Duplicates: 0 | Unique keys (seller_id): 3095


In [10]:
# 1. Nettoyage et conversion du type de poids
products['product_weight_g'] = pd.to_numeric(products['product_weight_g'], errors='coerce')

# 2. Enrichissement de order_items avec le poids
items_enriched = order_items.merge(
    products[['product_id', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']], 
    on='product_id', 
    how='left'
)

# 3. Agrégation au niveau order_id
items_agg = items_enriched.groupby('order_id').agg(
    total_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    max_shipping_limit_date=('shipping_limit_date', 'max'),
    avg_product_weight_g=('product_weight_g', 'mean'), # Fonctionnera parfaitement maintenant !
    seller_id=('seller_id', 'first')
).reset_index()

print("Agrégation réussie !")

Agrégation réussie !


In [ ]:
# 1. Jointure pour créer la table ML (1 ligne par commande)
ml_df = orders[orders['order_status'] == 'delivered'].copy()
ml_df = ml_df.merge(customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']], on='customer_id', how='left')
ml_df = ml_df.merge(items_agg, on='order_id', how='left')
ml_df = ml_df.merge(payments_agg, on='order_id', how='left')
ml_df = ml_df.merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']], on='seller_id', how='left')




✅ Artefact généré avec succès (96478 lignes) : artifacts/01_joined_ml_table.parquet


In [ ]:
# 1. Validation et Sauvegarde de l'Artefact N°1
assert len(ml_df) == ml_df['order_id'].nunique(), "Erreur : Doublons détectés dans order_id !"

output_path = "artifacts/01_joined_ml_table.parquet"
ml_df.to_parquet(output_path, index=False)

print(f"\n✅ Artefact généré avec succès ({len(ml_df)} lignes) : {output_path}")